# Stage C — T4 horizon-3 handoff
Asserts a T4 before spending compute, runs contract tests, then captures FP32/FP16 horizon-3 capacity evidence.

In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='04fba759f724b007ee55737dd0ad3bb29fa37530'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
DATASET_DIR=f'{DRIVE_ROOT}/datasets/ordered_streams/LOCKED_TOKENIZER'
RUN_NAME='c5_t4_horizon3'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'remote','set-url','origin',REPO_URL],check=True)
subprocess.run(['git','-C',str(repo),'fetch','--all'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
output=f'{DRIVE_ROOT}/runs/{RUN_NAME}'
Path(output).mkdir(parents=True,exist_ok=True)
def logged(label,command):
    subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',output,'--label',label,'--repo',str(repo),'--',*command],check=True)
logged('hardware_preflight',['seqtrainer-titans-stage-c-hardware-preflight','--require','T4','--output',f'{output}/hardware.json'])
logged('stage_c_tests',[sys.executable,'-m','pytest','-q',str(repo/'tests/test_titans_paper_mac_stage_c_tokenizers.py'),str(repo/'tests/test_titans_paper_mac_stage_c_model.py')])

In [ ]:
logged('t4_horizon3_capacity',['seqtrainer-titans-stage-c-capacity','--dataset-dir',DATASET_DIR,'--output-dir',output,'--require','T4','--horizons','3','--variants','reference_fp32','exact_sdpa_fp32','exact_sdpa_float16','--steps','2','--batch-size','1'])
print('SHARE THIS DIRECTORY:',output)